# Fifa Prediction model

Each training example will be a (22,9) matrix. Each row is a player, the first 11 rows are players from the home team while the last 11 rows are players from the away team. For each player, the following details are used:

1. age
2. is forward
3. is mid fielder
4. is defense
5. is goalkeeper
6. number tournaments
7. appearances
8. goals
9. average time per team

This matrix will be passed to the first layer of the neural network that should predict the 'quality' of each player in the team. There will be a ReLU activation for this network. This will be done by doing:

$$
p = xw^T + b \\
ap = max(0, p)
$$

The output of this will be a (22, 1) matrix. This will then be passed to the team wins prediction portion of the network. This has 1 layer with 121 units with softmax activation as shown below:

$$
L = 1 \\
n^{[1]} = 2 \\
z = apw + b \\
y = softmax(z)
$$

The goal is to get a model that surpasses the following benchmarks:
- 75.42% accuracy on predicting a win
- 18% accuracy for predicting exact scores

In [30]:
# Imports
import numpy as np
import numpy.typing as npt
from sklearn.model_selection import train_test_split
import math

In [15]:
def f_x(
    players: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float
):
    """ 
    Function that will do a forward pass for a single training example in the example above.
    
    Args:
        players (ndarray) - a (22,9) numpy array with 9 features for 22 players from the home and away team
        w_p (ndarray) - a (1,9) numpy array with weights for player predictions
        b_p (scalar) - bias for player quality prediction
        w_t (ndarray) - a (121, 22) numpy array with the weights for the team score prediction
        b_t (scalar) - bias for the team score prediction
        
    Returns:
        scores (ndarray) - a (121,1) numpy array with probability of each of the score distributions
        z_t (ndarray) - cached z_t value
        a_p (ndarray) - cached a_t value,
        z_p (ndarray) - cached z_p value
    """
    z_p = np.matmul(players, w_p.T) + b_p
    a_p = np.maximum(0, z_p)
    z_t = np.matmul(w_t, a_p) + b_t
    shifted_logits = z_t - np.max(z_t, axis=0, keepdims=True)
    e_zi = np.exp(shifted_logits)
    a_t = e_zi / np.sum(e_zi, axis=0, keepdims=True)
    return (a_t, z_t, a_p, z_p)
    

In [16]:
# Forward pass is working
players_1 = np.ones((1, 9))
players_1 = np.tile(players_1, (22, 1))
w_p = np.ones((1, 9))
b_p = 0
w_t = np.ones((121, 22))
b_t = 0
result_1 = f_x(
    players_1,
    w_p,
    b_p,
    w_t,
    b_t
)
print(result_1[0])

[[0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00826446]
 [0.00

In [18]:
def L(y_pred, y):
    """
    Returns the means squared error of a single example
    
    Args:
        y_pred (ndarray) - predicted home away goal scores
        y (ndarray) - actual home away goal scores
        
    Returns:
        error (scalar) - mean squared error loss
    """
    loss = np.where(y == 1, -np.log(y_pred + 1e-15), 0) # Adding by a small number to prevent trying to get log of 0
    return np.sum(loss)

In [19]:
print(L(np.array([0.778,0.232]), np.array([1,0])))

0.25102875480374415


In [20]:
def calculate_gradients(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray
):
    """ 
    Function to calculate the gradients used in gradient descent
    
    Args:
        X (ndarray) - a (m,22,9) array with m training examples
        Y (ndarray) - a (m, 121, 1) array with labels for m training examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
        
    Returns:
        gradients (list) - list of gradients of parameters
    """
    dw_t = np.zeros(w_t.shape)
    db_t = 0
    dw_p = np.zeros(w_p.shape)
    db_p = 0
    J = 0
    
    m = X.shape[0]
    
    # For each training example
    for i in range(m):
        # Get the prediction
        x_i = X[i]
        y_i = Y[i]
        (y_pred_i, z_t_i, a_p_i, z_p_i) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        
        
        # Get derivative
        # da_t_i = 2 * (y_pred_i - y_i)
        dz_t_i = y_pred_i - y_i
        dw_t += a_p_i.T * dz_t_i
        db_t += np.sum(dz_t_i)
        da_p_i = np.matmul(w_t.T, dz_t_i)
        dz_p_i = np.where(z_p_i < 0, 0, da_p_i)
        dw_p += np.matmul(dz_p_i.T, x_i)
        db_p += np.sum(dz_p_i)
        
        # Add loss 
        J += L(y_pred_i, y_i)
        
        
    dw_t /= m
    db_t /= m
    dw_p /= m
    db_p /= m
    J /= m
    return (dw_p, db_p, dw_t, db_t, J)

In [21]:
x_train_1 = np.ones((1, 9))
x_train_1 = np.tile(x_train_1, (22, 1))
w_p = np.ones((1, 9))
b_p = 0
w_t = np.ones((121, 22))
b_t = 0
X_train = np.array([
    x_train_1
])
Y_train = np.zeros((121, 1))
Y_train[0][0] = 1

print(calculate_gradients(
    X_train,
    w_p,
    b_p,
    w_t,
    b_t,
    Y_train
))

(array([[-2640., -2640., -2640., -2640., -2640., -2640., -2640., -2640.,
        -2640.]]), np.float64(-2639.9999999999986), array([[-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       ...,
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983],
       [-8.92561983, -8.92561983, -8.92561983, ..., -8.92561983,
        -8.92561983, -8.92561983]], shape=(121, 22)), np.float64(-120.00000000000001), np.float64(580.2906560171909))


In [22]:
# Splitting data into train / cross-validation / test sets
X = np.load("data/players_batch_6_2026-05-25 21:44:14.801126.npy")
Y = np.load("data/results_batch_6_2026-05-25 21:44:14.801174.npy")
X_train, X_mid, Y_train, Y_mid = train_test_split(X, Y, test_size=0.2, random_state=42)
X_cv, X_test, Y_cv, Y_test = train_test_split(X_mid, Y_mid, test_size=0.5, random_state=42)
print(X_train.shape, X_cv.shape, X_test.shape)
print(Y_train.shape, Y_cv.shape, Y_test.shape)

(560, 22, 9) (70, 22, 9) (70, 22, 9)
(560, 121, 1) (70, 121, 1) (70, 121, 1)


In [23]:
def gradient_descent(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray,
    alpha: float,
    num_iters: int
):
    """ 
    This function will minimize the cost of the model using gradient descent and return found weights
    
    Args:
        X (ndarray) - a (m, 22, 9) array serving as input to model
        w_p (ndarray) - a (1,9) array with the initial weights of the player section of model
        b_p (scalar) - initial bias for player section of model
        w_t (ndarray) - a (2,22) array with initial weights of the team section of model
        b_t (scalar) - initial bias for team section of model
        Y (ndarray) - a (m, 2, 1) array with target outputs for model
        alpha (scalar) - learning rate
        num_iters (scalar) - number of times to run gradient descent
        
    Return:
        w_p (ndarray) - a (1, 9) array that minimizes cost
        b_p (scalar) - bias that minimized cost
        w_t (ndarray) - a (2, 22) array that minimizes cost
        b_t (scalar) - bias that minimizes cost
    """
    for i in range(num_iters):
        # Get derivatives
        (dw_p, db_p, dw_t, db_t, J) = calculate_gradients(
            X=X,
            w_p=w_p,
            b_p=b_p,
            w_t=w_t,
            b_t=b_t,
            Y=Y
        )
        
        # Update weights
        w_p = w_p - (alpha * dw_p)
        w_t = w_t - (alpha * dw_t)
        b_p = b_p - (alpha * db_p)
        b_t = b_t - (alpha * db_t)
        
        # Print result
        print(f"Cost at epoch {i}/{num_iters} = {J}")
        
    return (w_p, w_t, b_p, b_t)

In [24]:
# Get initial weights
w_t = np.random.rand(121, 22)
w_p = np.random.rand(1, 9)
b_p = 0
b_t = 0

# Run gradient descent
(new_w_p, new_w_t, new_b_p, new_bt) = gradient_descent(
    X=X_train,
    w_p=w_p,
    w_t=w_t,
    b_p=b_p,
    b_t=b_t,
    Y=Y_train,
    alpha=0.01,
    num_iters=100
)

Cost at epoch 0/100 = 34.538776394911
Cost at epoch 1/100 = 4.795790545596609
Cost at epoch 2/100 = 4.795790545596609
Cost at epoch 3/100 = 4.795790545596609
Cost at epoch 4/100 = 4.795790545596609
Cost at epoch 5/100 = 4.795790545596609
Cost at epoch 6/100 = 4.795790545596609
Cost at epoch 7/100 = 4.795790545596609
Cost at epoch 8/100 = 4.795790545596609
Cost at epoch 9/100 = 4.795790545596609
Cost at epoch 10/100 = 4.795790545596609
Cost at epoch 11/100 = 4.795790545596609
Cost at epoch 12/100 = 4.795790545596609
Cost at epoch 13/100 = 4.795790545596609
Cost at epoch 14/100 = 4.795790545596609
Cost at epoch 15/100 = 4.795790545596609
Cost at epoch 16/100 = 4.795790545596609
Cost at epoch 17/100 = 4.795790545596609
Cost at epoch 18/100 = 4.795790545596609
Cost at epoch 19/100 = 4.795790545596609
Cost at epoch 20/100 = 4.795790545596609
Cost at epoch 21/100 = 4.795790545596609
Cost at epoch 22/100 = 4.795790545596609
Cost at epoch 23/100 = 4.795790545596609
Cost at epoch 24/100 = 4.795

In [25]:
def evaluate(
    X: npt.NDArray, 
    w_p: npt.NDArray, 
    b_p: float, 
    w_t: npt.NDArray, 
    b_t: float,
    Y: npt.NDArray
):
    """ 
    Evaluate model on dataset
    
    Args:
        X (ndarray) - a (m,22,9) array with m examples
        Y (ndarray) - a (m, 2, 1) array with labels for m examples
        w_p (ndarray) - weights for the player section of neural network
        b_p (scalar) - bias for the player section of neural network
        w_t (ndarray) - weights for team section of neural network
        b_t (scalar) - bias for team section of neural network
    """
    J = 0
    m = X.shape[0]
    
    # For each training example
    for i in range(m):
        # Get the prediction
        x_i = X[i]
        y_i = Y[i]
        (y_pred_i, _, _, _) = f_x(
            x_i,
            w_p,
            b_p,
            w_t,
            b_t
        )
        
        # Add loss 
        J += L(y_pred_i, y_i)
        
    J /= m
    print(f"Cost is {J}")
    

In [26]:
evaluate(
    X=X_cv,
    w_p=new_w_p,
    b_p=new_b_p,
    w_t=new_w_t,
    b_t=new_bt,
    Y=Y_cv
)

Cost is 4.795790545596618


In [40]:
def interpret_probabilities(prob):
    """ 
    Function to interpret the returned probabilities by the model
    """
    predicted = np.argmax(prob)
    home_score = math.floor(predicted / 11)
    away_score = predicted % 11
    return f"{home_score}-{away_score}"

In [41]:
prob = f_x(
    X_cv[5],
    new_w_p,
    new_b_p,
    new_w_t,
    new_bt
)[0]

predicted_score = interpret_probabilities(prob)
actual_score = interpret_probabilities(Y_cv[5])
print(f"Predicted score is {predicted_score} while actual is {actual_score}")

Predicted score is 0-0 while actual is 5-0
